# 933. Number of Recent Calls

**Difficulty**: Easy  
**Topics**: Design, Queue, Data Stream  
**Link**: [LeetCode 933](https://leetcode.com/problems/number-of-recent-calls/)

---

## Problem Statement

You have a `RecentCounter` class which counts the number of recent requests within a certain time frame.

Implement the `RecentCounter` class:

- `RecentCounter()` Initializes the counter with zero recent requests.
- `int ping(int t)` Adds a new request at time `t`, where `t` represents some time in milliseconds, and returns the number of requests that has happened in the past `3000` milliseconds (including the new request). Specifically, return the number of requests that have happened in the inclusive range `[t - 3000, t]`.

It is **guaranteed** that every call to `ping` uses a strictly larger value of `t` than the previous call.

### Examples

**Example 1:**
```
Input:
["RecentCounter", "ping", "ping", "ping", "ping"]
[[], [1], [100], [3001], [3002]]

Output:
[null, 1, 2, 3, 3]

Explanation:
RecentCounter recentCounter = new RecentCounter();
recentCounter.ping(1);     // requests = [1], range is [-2999,1], return 1
recentCounter.ping(100);   // requests = [1, 100], range is [-2900,100], return 2
recentCounter.ping(3001);  // requests = [1, 100, 3001], range is [1,3001], return 3
recentCounter.ping(3002);  // requests = [1, 100, 3001, 3002], range is [2,3002], return 3
```

### Constraints

- `1 <= t <= 10^9`
- Each test case will call `ping` with **strictly increasing** values of `t`.
- At most `10^4` calls will be made to `ping`.

---

## Approach: Queue (Sliding Window)

### Intuition

Since we only care about requests in the last 3000ms, we can use a **queue** to maintain a sliding window:

1. Add new request time to the queue
2. Remove all requests older than `t - 3000` from the front
3. Return the queue size

### Why Queue?

- Requests come in **strictly increasing** order
- Old requests are always at the **front** of the queue
- We only need to remove from front (FIFO)

### Complexity
- **Time**: O(1) amortized per ping (each element added/removed at most once)
- **Space**: O(W) where W = 3000 (max window size)

In [ ]:
from collections import deque

class RecentCounter:
    """Queue-based solution: O(1) amortized time per ping"""
    
    def __init__(self):
        self.requests = deque()
    
    def ping(self, t: int) -> int:
        # Add new request
        self.requests.append(t)
        
        # Remove requests outside the window [t-3000, t]
        while self.requests[0] < t - 3000:
            self.requests.popleft()
        
        return len(self.requests)

# Test
counter = RecentCounter()
print(counter.ping(1))     # 1
print(counter.ping(100))   # 2
print(counter.ping(3001))  # 3
print(counter.ping(3002))  # 3

---

## Walkthrough

```
ping(1):    queue = [1]           range = [-2999, 1]    → 1 request
ping(100):  queue = [1, 100]      range = [-2900, 100]  → 2 requests
ping(3001): queue = [1, 100, 3001] range = [1, 3001]    → 3 requests
ping(3002): queue = [100, 3001, 3002] range = [2, 3002] → 3 requests
            (removed 1 because 1 < 3002-3000=2)
```

In [ ]:
class RecentCounterVerbose:
    """Verbose version showing the process"""
    
    def __init__(self):
        self.requests = deque()
        print("RecentCounter initialized")
    
    def ping(self, t: int) -> int:
        self.requests.append(t)
        window_start = t - 3000
        
        print(f"\nping({t}):")
        print(f"  Window: [{window_start}, {t}]")
        print(f"  Queue before cleanup: {list(self.requests)}")
        
        removed = []
        while self.requests[0] < window_start:
            removed.append(self.requests.popleft())
        
        if removed:
            print(f"  Removed (too old): {removed}")
        
        print(f"  Queue after cleanup: {list(self.requests)}")
        print(f"  Count: {len(self.requests)}")
        
        return len(self.requests)

# Test
counter = RecentCounterVerbose()
counter.ping(1)
counter.ping(100)
counter.ping(3001)
counter.ping(3002)

---

## Alternative: List with Binary Search

If we need to keep all requests (not just recent ones), we can use binary search to find the count.

### Complexity
- **Time**: O(log n) per ping
- **Space**: O(n) - stores all requests

In [ ]:
import bisect

class RecentCounterBinarySearch:
    """Binary search approach: O(log n) per ping"""
    
    def __init__(self):
        self.requests = []
    
    def ping(self, t: int) -> int:
        self.requests.append(t)
        
        # Find first index >= t - 3000
        window_start = t - 3000
        left_idx = bisect.bisect_left(self.requests, window_start)
        
        # Count = total - elements before window
        return len(self.requests) - left_idx

# Test
counter = RecentCounterBinarySearch()
print(counter.ping(1))     # 1
print(counter.ping(100))   # 2
print(counter.ping(3001))  # 3
print(counter.ping(3002))  # 3

---

## Comparison of Approaches

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Queue** | O(1) amortized | O(W) | Fast, memory efficient | - |
| **Binary Search** | O(log n) | O(n) | Keeps all history | Uses more memory |

---

## Common Mistakes

1. **Using a list instead of deque** - `list.pop(0)` is O(n), `deque.popleft()` is O(1)
2. **Off-by-one in window** - range is `[t-3000, t]` inclusive
3. **Not handling edge case** - first ping always returns 1

---

## Related Problems

- [346. Moving Average from Data Stream](https://leetcode.com/problems/moving-average-from-data-stream/) - Easy
- [362. Design Hit Counter](https://leetcode.com/problems/design-hit-counter/) - Medium
- [239. Sliding Window Maximum](https://leetcode.com/problems/sliding-window-maximum/) - Hard

---

## Key Takeaways

| Pattern | When to Use |
|---------|-------------|
| **Queue for sliding window** | Time-based windows with monotonic input |
| **Deque over list** | Need O(1) removal from front |
| **Binary search** | When you need to keep full history |